# Whisper large-v3-turbo — DIMER LoRA fine-tuning tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/whisper-asr-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/whisper-asr-pipeline/blob/main/tutorials/whisper_asr_finetune_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-openai%2Fwhisper--large--v3--turbo-ffcc4d?style=flat)](https://huggingface.co/openai/whisper-large-v3-turbo)
[![Upstream](https://img.shields.io/badge/Upstream-openai%2Fwhisper-181717?style=flat&logo=github&logoColor=white)](https://github.com/openai/whisper)
[![arXiv](https://img.shields.io/badge/arXiv-2212.04356-b31b1b.svg)](https://arxiv.org/abs/2212.04356)

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** parameter-efficient (LoRA) domain adaptation of the pinned OpenAI Whisper large-v3-turbo weights for automatic speech recognition, evaluated by word error rate before and after adaptation

This notebook is the executable reference path for adapting the repository model to a small labelled speech set. It exercises the repository's public pipeline API for model resolution, zero-shot evaluation, and adapter reload; only the training loop itself lives in the notebook, in plain PyTorch, so every moving part is visible. The default sample is demonstration evidence, not a production-quality or benchmark claim.

**Learning objectives:** bootstrap the repository in a fresh runtime, resolve the immutable upstream model revision, load and resample a public labelled speech set (or an optional BYOD set), record a zero-shot WER baseline through the public API, attach LoRA adapters and train them with mixed precision, measure the adapted WER on a held-out split, export a manifested adapter bundle, and prove the bundle reloads through the public API with the same effect.


## Prerequisites

Run in a fresh supported runtime with a CUDA GPU (a free Colab or Kaggle T4 is sufficient for the default configuration; the default path needs roughly 6 GiB of GPU memory and 10–15 minutes). CPU execution is supported only as a reduced smoke test: lower `TRAIN_CLIPS`, `EVAL_CLIPS` and `EPOCHS` first, or the training cell will take hours. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.


## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package with its `tutorial` and `finetune` extras so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies and `peft` are directly pinned. If installation replaces any package that this runtime has already imported, the cell fails with a restart instruction rather than continuing with mixed versions.


In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/whisper-asr-pipeline.git'
REPO_NAME = 'whisper-asr-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.6.0+cu124) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{ROOT}[tutorial,finetune]'], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, torch, transformers, peft
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'cuda': torch.cuda.is_available()})


## 2. Load a public labelled speech set or optional BYOD

The default path loads one locale of the public `PolyAI/minds14` banking-intent corpus (CC-BY-4.0): short telephone-quality utterances stored at **8 kHz**, each with a cased, punctuated transcription. Whisper expects 16 kHz input, so every clip is decoded with the pinned `soundfile` dependency and resampled with the pinned `torchaudio` — the same resampler the public pipeline applies at inference time, so training and inference see identical audio. A seeded shuffle then takes `TRAIN_CLIPS` clips for adaptation and a disjoint `EVAL_CLIPS` for the held-out baseline/adapted comparison; the split is recorded as a digest in the exported provenance.

The locale list is restricted to languages whose transcripts are whitespace-delimited: the package's word error rate splits on whitespace, so it is not a meaningful metric for the corpus's `zh-CN` and `ko-KR` locales (a character-level metric would be needed there).

BYOD is optional and disabled by default. Expected BYOD input is a `transcripts.csv` with `file,text` columns plus the referenced audio files (WAV/FLAC/OGG readable by `soundfile`), all selected in the same upload dialog; `BYOD_LANGUAGE` is the ISO-639-1 code of the uploaded speech and the same whitespace caveat applies.


In [ ]:
import csv
import hashlib
import io
import random

import soundfile as sf
import torchaudio
from datasets import Audio, load_dataset

LOCALE = 'en-US'  # @param ['en-US', 'en-GB', 'en-AU', 'de-DE', 'fr-FR', 'es-ES', 'it-IT', 'nl-NL', 'pl-PL', 'pt-PT', 'ru-RU', 'cs-CZ']
TRAIN_CLIPS = 400  # @param {type:"integer"}
EVAL_CLIPS = 100  # @param {type:"integer"}
SEED = 0  # @param {type:"integer"}
USE_BYOD = False  # @param {type:"boolean"}
BYOD_LANGUAGE = 'en'  # @param {type:"string"}
TARGET_RATE = 16_000


def decode_clip(raw_bytes):
    # Decode with the pinned soundfile, then resample with the pinned torchaudio: the datasets
    # audio feature would decode through torchcodec/FFmpeg, which this runtime does not pin.
    waveform, rate = sf.read(io.BytesIO(raw_bytes), dtype='float32')
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    if rate != TARGET_RATE:
        waveform = torchaudio.functional.resample(torch.from_numpy(waveform), rate, TARGET_RATE).numpy()
    return waveform


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    manifest_name = next(name for name in uploaded if name.lower().endswith('.csv'))
    rows = list(csv.DictReader(io.StringIO(uploaded[manifest_name].decode('utf-8-sig'))))
    clips = [{'id': row['file'], 'audio': decode_clip(uploaded[row['file']]), 'text': row['text'].strip()} for row in rows]
    LANGUAGE = BYOD_LANGUAGE.strip().lower()
    DATASET = {'name': 'BYOD', 'config': manifest_name, 'license': None}
else:
    ds = load_dataset('PolyAI/minds14', LOCALE, split='train')
    ds = ds.cast_column('audio', Audio(decode=False))
    clips = [{'id': row['path'], 'audio': decode_clip(row['audio']['bytes']), 'text': row['transcription'].strip()} for row in ds]
    LANGUAGE = LOCALE.split('-')[0]
    DATASET = {'name': 'PolyAI/minds14', 'config': LOCALE, 'license': 'CC-BY-4.0'}

if TRAIN_CLIPS < 1 or EVAL_CLIPS < 1 or TRAIN_CLIPS + EVAL_CLIPS > len(clips):
    raise ValueError(f'TRAIN_CLIPS + EVAL_CLIPS must fit in the {len(clips)} available clips')
order = list(range(len(clips)))
random.Random(SEED).shuffle(order)
train_clips = [clips[i] for i in order[:TRAIN_CLIPS]]
eval_clips = [clips[i] for i in order[TRAIN_CLIPS:TRAIN_CLIPS + EVAL_CLIPS]]
SPLIT_DIGEST = hashlib.sha256('\n'.join(c['id'] + '\t' + c['text'] for c in train_clips + eval_clips).encode('utf-8')).hexdigest()
seconds = [len(c['audio']) / TARGET_RATE for c in train_clips + eval_clips]
print({'dataset': DATASET, 'language': LANGUAGE, 'train_clips': len(train_clips), 'eval_clips': len(eval_clips), 'split_digest': SPLIT_DIGEST[:16], 'seconds_mean': round(sum(seconds) / len(seconds), 2), 'seconds_max': round(max(seconds), 2)})
print({'example_text': train_clips[0]['text']})


## 3. Resolve the pinned model and record the zero-shot baseline

The public API pins the exact upstream revision and refuses remote model code. Before anything is trained, the unmodified model transcribes the held-out clips through `WhisperASRPipeline.transcribe`, and the corpus WER (total word edits over total reference words, after the package's case and punctuation normalization) is recorded as the **baseline**. The pipeline is then released so the training model has the GPU to itself.


In [ ]:
import gc
import time

from whisper_asr_pipeline import MODEL_ID, MODEL_REVISION, WhisperASRPipeline, load_model, word_error_count, word_error_rate
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})


def corpus_wer(references, hypotheses):
    counts = [word_error_count(reference, hypothesis) for reference, hypothesis in zip(references, hypotheses)]
    return sum(errors for errors, _ in counts) / sum(length for _, length in counts)


def transcribe_all(pipe, clip_list):
    return [pipe.transcribe({'array': c['audio'], 'sampling_rate': TARGET_RATE}, language=LANGUAGE, task='transcribe')['text'] for c in clip_list]


eval_references = [c['text'] for c in eval_clips]
pipe = WhisperASRPipeline.from_pretrained()
DEVICE = pipe.device
started = time.time()
baseline_transcripts = transcribe_all(pipe, eval_clips)
baseline_wer = corpus_wer(eval_references, baseline_transcripts)
print({'device': DEVICE, 'baseline_wer': round(baseline_wer, 4), 'seconds': round(time.time() - started, 1)})
for reference, hypothesis in list(zip(eval_references, baseline_transcripts))[:3]:
    print({'reference': reference, 'baseline': hypothesis, 'wer': round(word_error_rate(reference, hypothesis), 3)})
del pipe
gc.collect()
if DEVICE.startswith('cuda'):
    torch.cuda.empty_cache()


## 4. Attach LoRA adapters

**LoRA** keeps every original weight matrix $W_0$ frozen and learns a low-rank correction $W = W_0 + \frac{\alpha}{r} B A$; $B$ starts at zero, so at step 0 the adapted model *is* the base model. Adapters go on the query and value projections of every attention block in both the encoder (32 layers) and the 4-layer decoder, which is the standard Whisper recipe: with rank 32 that is under 1% of the 809M parameters. The base model is loaded through the public API, promoted to float32 master weights (mixed precision runs the matmuls in float16), and gradient checkpointing is enabled so the encoder's 30-second activations fit a 16 GiB card at batch size 4. The forced decoder prompt is cleared so training and generation both derive the language/task prefix from the labels and the `generate` arguments rather than a config default.


In [ ]:
from peft import LoraConfig, get_peft_model

LORA_RANK = 32  # @param {type:"integer"}
LORA_ALPHA = 64  # @param {type:"integer"}
LORA_TARGETS = ['q_proj', 'v_proj']

base_model, processor = load_model(device=DEVICE)
base_model = base_model.float()
# One frozen weight, kept as it was loaded, so the reload step can prove the saved adapter is
# applied to the fresh weights (W_reloaded == W_base + (alpha / r) * B @ A) independently of
# how much the transcripts moved.
PROBE_MODULE = 'model.decoder.layers.0.self_attn.q_proj'
probe_base_weight = base_model.get_submodule(PROBE_MODULE).weight.detach().to('cpu', torch.float32).clone()
base_model.config.forced_decoder_ids = None
base_model.generation_config.forced_decoder_ids = None
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})

lora_config = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias='none', target_modules=LORA_TARGETS)
model = get_peft_model(base_model, lora_config)
trainable_params, all_params = model.get_nb_trainable_parameters()
print({'lora_targets': LORA_TARGETS, 'trainable_parameters': trainable_params, 'all_parameters': all_params, 'trainable_percent': round(100 * trainable_params / all_params, 3), 'dtype': str(next(model.parameters()).dtype)})


## 5. Train

The loop is deliberately plain PyTorch rather than a `Trainer`, so every moving part is visible:

- **Features and labels.** Each clip becomes a 128-bin log-mel spectrogram padded to Whisper's fixed 30-second window by the pinned processor. The label sequence is the tokenizer's rendering of the transcript with its language/task prefix, minus the leading start token (the model prepends it when it shifts labels right), padded with `-100` so padding is ignored by the loss.
- **Mixed precision.** On CUDA the forward pass runs under float16 autocast with a gradient scaler; the LoRA weights and optimizer state stay in float32. On CPU everything is float32.
- **AdamW at a constant learning rate** (`1e-3` is the usual LoRA starting point for Whisper) with `GRAD_ACCUM` micro-batches per optimizer step. There is no warmup or decay schedule here; for two epochs on a few hundred clips a schedule changes little, and leaving it out keeps the loop readable.
- **Validation loss** is the teacher-forced cross-entropy on the held-out clips after each epoch. It tracks whether the adapter is still learning; WER, the metric that matters, is measured in the next section.


In [ ]:
EPOCHS = 2  # @param {type:"integer"}
BATCH_SIZE = 4  # @param {type:"integer"}
GRAD_ACCUM = 2  # @param {type:"integer"}
LEARNING_RATE = 1e-3  # @param {type:"number"}

processor.tokenizer.set_prefix_tokens(language=LANGUAGE, task='transcribe')
decoder_start = model.config.decoder_start_token_id
use_amp = DEVICE.startswith('cuda')


def make_batch(clip_list):
    features = processor.feature_extractor([c['audio'] for c in clip_list], sampling_rate=TARGET_RATE, return_tensors='pt').input_features
    label_rows = []
    for c in clip_list:
        ids = processor.tokenizer(c['text']).input_ids
        if ids and ids[0] == decoder_start:
            ids = ids[1:]
        label_rows.append(ids)
    width = max(len(ids) for ids in label_rows)
    labels = torch.full((len(label_rows), width), -100, dtype=torch.long)
    for row, ids in enumerate(label_rows):
        labels[row, :len(ids)] = torch.tensor(ids)
    return features.to(DEVICE), labels.to(DEVICE)


def batches(clip_list, size):
    for start in range(0, len(clip_list), size):
        yield clip_list[start:start + size]


def evaluate_loss(clip_list):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.inference_mode():
        for batch in batches(clip_list, BATCH_SIZE):
            features, labels = make_batch(batch)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                loss = model(input_features=features, labels=labels).loss
            count = int((labels != -100).sum())
            total_loss += loss.item() * count
            total_tokens += count
    return total_loss / total_tokens


optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
shuffler = random.Random(SEED)
if use_amp:
    torch.cuda.reset_peak_memory_stats()
started = time.time()
history = []
OPTIMIZER_STEPS = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_clips = train_clips[:]
    shuffler.shuffle(epoch_clips)
    optimizer.zero_grad(set_to_none=True)
    epoch_loss, epoch_tokens, pending, epoch_steps = 0.0, 0, 0, 0
    micro_batches = list(batches(epoch_clips, BATCH_SIZE))
    for step, batch in enumerate(micro_batches, start=1):
        features, labels = make_batch(batch)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            loss = model(input_features=features, labels=labels).loss
        # The window is measured from its own first micro-batch, so the last (possibly short)
        # window is scaled by its true size and still takes its optimizer step.
        window_start = ((step - 1) // GRAD_ACCUM) * GRAD_ACCUM
        window = min(GRAD_ACCUM, len(micro_batches) - window_start)
        scaler.scale(loss / window).backward()
        count = int((labels != -100).sum())
        epoch_loss += loss.item() * count
        epoch_tokens += count
        pending += 1
        if pending == window:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            pending = 0
            epoch_steps += 1
    expected_steps = -(-len(micro_batches) // GRAD_ACCUM)
    if epoch_steps != expected_steps or pending:
        raise RuntimeError(f'epoch {epoch} took {epoch_steps} optimizer steps, expected {expected_steps} ({pending} micro-batches left unstepped)')
    OPTIMIZER_STEPS += epoch_steps
    train_loss = epoch_loss / epoch_tokens
    validation_loss = evaluate_loss(eval_clips)
    history.append({'epoch': epoch, 'train_loss': round(train_loss, 4), 'validation_loss': round(validation_loss, 4), 'optimizer_steps': epoch_steps})
    print(history[-1])
TRAINING_SECONDS = round(time.time() - started, 1)
PEAK_GPU_GIB = round(torch.cuda.max_memory_allocated() / 1024 ** 3, 2) if use_amp else None
print({'training_seconds': TRAINING_SECONDS, 'peak_gpu_gib': PEAK_GPU_GIB, 'optimizer_steps_total': OPTIMIZER_STEPS})


## 6. Evaluate the adapted model on the held-out clips

The live LoRA model is wrapped in the same public pipeline that produced the baseline (`WhisperASRPipeline.from_model`), so the adapted transcripts come from exactly the decoding path a user of the repository gets (the Transformers ASR pipeline decodes Whisper with 5-beam search and the model's 448-token limit) — not from a hand-rolled `generate` call with different search or length settings, which can disagree with the pipeline on near-tie clips. On CUDA the model is first cast to float16, the precision the pipeline serves at. The corpus WER is compared with the baseline from Section 3 and a few changed clips are shown side by side. This number is tutorial evidence for one small split; it says whether the adapter helped *here*, not how it generalizes.


In [ ]:
model.eval()
model.config.use_cache = True
if use_amp:
    model = model.half()
adapted = WhisperASRPipeline.from_model(model, processor, adapter='in-memory LoRA')
adapted_transcripts = transcribe_all(adapted, eval_clips)
adapted_wer = corpus_wer(eval_references, adapted_transcripts)
del adapted
metrics = {'baseline_wer': round(baseline_wer, 4), 'adapted_wer': round(adapted_wer, 4), 'wer_delta': round(adapted_wer - baseline_wer, 4), 'eval_clips': len(eval_clips), 'history': history}
print(metrics)
changed = [(r, b, a) for r, b, a in zip(eval_references, baseline_transcripts, adapted_transcripts) if b != a]
print({'clips_with_changed_transcript': len(changed)})
for reference, before, after in changed[:3]:
    print({'reference': reference, 'baseline': before, 'adapted': after})


## 7. Export the adapter bundle and prove it reloads through the public API

The deliverable is the **adapter bundle**, not a copy of the base model: `adapter_model.safetensors` (a few tens of MB), `adapter_config.json`, `metrics.json`, `provenance.json` (repository revision, base model identifier and immutable revision, dataset identity and split digest, hyperparameters, runtime), and an `artifact-manifest.json` listing every file with its size and SHA-256. The bundle is zipped under `outputs/`.

Then the notebook proves the bundle is usable from disk: the training model is released, `WhisperASRPipeline.from_pretrained(adapter_dir=...)` loads a fresh pinned base and merges the saved adapter, and the held-out clips are transcribed again through the same pipeline. The run fails unless the saved LoRA weights are non-zero, the reloaded weight of a probe module equals the base weight plus the scaled `B @ A` read back from the bundle (the reloaded model exposes its weights through `.model`; this proves the adapter was applied to the fresh weights however small its effect on the transcripts), and the reloaded transcripts agree with the in-memory adapted transcripts on at least 95% of the held-out clips (merging the adapter into the weights rounds it slightly, which can flip a near-tie token on an occasional clip; a broken bundle fails on most of them). Both WER figures and the number of held-out clips whose transcript moved away from the baseline are recorded; the adapter's SHA-256 is carried in every transcription result. Finally the machine-readable result is written with every number above and the repository/model provenance.


In [ ]:
import json
import math
import zipfile

from safetensors.torch import load_file

os.makedirs('outputs', exist_ok=True)
ADAPTER_DIR = Path('outputs/whisper-asr-lora-adapter')
ARTIFACT_ZIP = Path('outputs/whisper-asr-lora-adapter.zip')
shutil.rmtree(ADAPTER_DIR, ignore_errors=True)
model.save_pretrained(str(ADAPTER_DIR), safe_serialization=True)


def sha256_of_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


PROVENANCE = {
    'artifactFormat': 'peft_adapter',
    'artifactFormatVersion': 1,
    'repository_revision': REPO_SHA,
    'baseModel': MODEL_ID,
    'baseModelRevision': MODEL_REVISION,
    'trustRemoteCode': False,
    'dataset': {**DATASET, 'language': LANGUAGE, 'train_clips': len(train_clips), 'eval_clips': len(eval_clips), 'seed': SEED, 'split_digest': SPLIT_DIGEST},
    'training': {'method': 'lora', 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM, 'learning_rate': LEARNING_RATE, 'lora_rank': LORA_RANK, 'lora_alpha': LORA_ALPHA, 'target_modules': LORA_TARGETS, 'mixed_precision': 'float16' if use_amp else None, 'seconds': TRAINING_SECONDS, 'peak_gpu_gib': PEAK_GPU_GIB},
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'device': DEVICE},
}
(ADAPTER_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(ADAPTER_DIR / 'provenance.json').write_text(json.dumps(PROVENANCE, indent=2), encoding='utf-8')
manifest = [{'path': p.relative_to(ADAPTER_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of_file(p)} for p in sorted(ADAPTER_DIR.rglob('*')) if p.is_file()]
(ADAPTER_DIR / 'artifact-manifest.json').write_text(json.dumps({'format': 'peft_adapter', 'formatVersion': 1, 'files': manifest}, indent=2), encoding='utf-8')
with zipfile.ZipFile(ARTIFACT_ZIP, 'w', zipfile.ZIP_STORED) as archive:
    for p in sorted(ADAPTER_DIR.rglob('*')):
        if p.is_file():
            archive.write(p, p.relative_to(ADAPTER_DIR).as_posix())
saved_b = [t for name, t in load_file(str(ADAPTER_DIR / 'adapter_model.safetensors')).items() if 'lora_B' in name]
if not saved_b or max(t.abs().max().item() for t in saved_b) == 0:
    raise RuntimeError('Saved adapter weights are zero or missing')
print({'adapter_dir': str(ADAPTER_DIR), 'files': len(manifest), 'zip_mib': round(ARTIFACT_ZIP.stat().st_size / 1024 ** 2, 1), 'zip_sha256': sha256_of_file(ARTIFACT_ZIP)})

del model, base_model, optimizer, scaler
gc.collect()
if use_amp:
    torch.cuda.empty_cache()
reloaded = WhisperASRPipeline.from_pretrained(adapter_dir=ADAPTER_DIR)

# Weight-level proof that the fresh load applied the saved adapter: for the probe module,
# W_reloaded must equal W_base + (alpha / r) * B @ A within the precision of the reloaded weights.
saved = load_file(str(ADAPTER_DIR / 'adapter_model.safetensors'))
lora_a = saved[f'base_model.model.{PROBE_MODULE}.lora_A.weight'].to(torch.float32)
lora_b = saved[f'base_model.model.{PROBE_MODULE}.lora_B.weight'].to(torch.float32)
expected_weight = probe_base_weight + (LORA_ALPHA / LORA_RANK) * (lora_b @ lora_a)
reloaded_weight = reloaded.model.get_submodule(PROBE_MODULE).weight.detach().to('cpu', torch.float32)
weight_tolerance = 1e-3 if reloaded.model.dtype == torch.float16 else 1e-5
adapter_delta = (expected_weight - probe_base_weight).abs().max().item()
merge_error = (reloaded_weight - expected_weight).abs().max().item()
if adapter_delta <= 4 * weight_tolerance:
    raise RuntimeError(f'Adapter delta {adapter_delta:.2e} on {PROBE_MODULE} is below the {weight_tolerance:.0e} verification resolution; train longer or with a higher learning rate')
if merge_error > weight_tolerance:
    raise RuntimeError(f'Reloaded weights differ from base + scaled B@A by {merge_error:.2e} (> {weight_tolerance:.0e}) on {PROBE_MODULE}')

reloaded_transcripts = transcribe_all(reloaded, eval_clips)
reloaded_wer = corpus_wer(eval_references, reloaded_transcripts)
agreement = sum(a == r for a, r in zip(adapted_transcripts, reloaded_transcripts))
if agreement < math.ceil(0.95 * len(eval_clips)):
    raise RuntimeError(f'Reloaded adapter reproduces only {agreement}/{len(eval_clips)} in-memory adapted transcripts')
metrics['reloaded_wer'] = round(reloaded_wer, 4)
metrics['reload_agreement'] = f'{agreement}/{len(eval_clips)}'
metrics['clips_changed_vs_baseline'] = sum(b != r for b, r in zip(baseline_transcripts, reloaded_transcripts))
metrics['reload_weight_check'] = {'module': PROBE_MODULE, 'adapter_delta_max': adapter_delta, 'merge_error_max': merge_error, 'tolerance': weight_tolerance}
print({'reloaded_wer': metrics['reloaded_wer'], 'reload_agreement': metrics['reload_agreement'], 'clips_changed_vs_baseline': metrics['clips_changed_vs_baseline'], 'adapter': reloaded.adapter, 'adapter_sha256': reloaded.adapter_sha256, 'reload_weight_check': metrics['reload_weight_check']})

payload = {
    'metrics': metrics,
    'examples': [{'reference': r, 'baseline': b, 'reloaded': a} for r, b, a in list(zip(eval_references, baseline_transcripts, reloaded_transcripts))[:5]],
    'artifact': {'zip': str(ARTIFACT_ZIP), 'sha256': sha256_of_file(ARTIFACT_ZIP), 'files': manifest},
    'provenance': PROVENANCE,
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': reloaded.device,
    },
    'sample': 'BYOD' if USE_BYOD else f'PolyAI/minds14:{LOCALE}',
}
with open('outputs/whisper_asr_finetune_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print('outputs/whisper_asr_finetune_result.json')


## Interpretation and limits

The adapted transcripts are model-generated. The baseline, adapted and reloaded WER figures are corpus-level numbers on one small held-out split of one locale of one public corpus (or of the uploaded BYOD set) and must not be generalized to other languages, speakers, domains, or capture conditions; a WER that moved by a few points on 100 short utterances is within the range where a different seed can change the sign. The adapter specializes the model toward the training distribution and can degrade it elsewhere (catastrophic forgetting); the tutorial measures nothing outside the held-out split. The pipeline provides no diarization, speaker identity, biometric inference, or calibrated transcript-confidence threshold, and the adapter inherits the license obligations of both the base weights (MIT) and the training data (CC-BY-4.0 for the default sample).

Successful execution proves that the recorded repository revision can acquire the pinned model, load and resample the demonstrated data, record a zero-shot baseline through the public API, train LoRA adapters with the shown configuration, export a manifested adapter bundle, and reload that bundle through the public API with a measurable, reproduced effect in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Inference tutorial: `whisper_asr_colab.ipynb`
- Upstream model: https://huggingface.co/openai/whisper-large-v3-turbo
- Whisper paper: https://arxiv.org/abs/2212.04356
- LoRA paper: https://arxiv.org/abs/2106.09685
- Training data: https://huggingface.co/datasets/PolyAI/minds14
